In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np




In [ ]:
class MinimalCNN(nn.Module):
# conv1, fc and pooling

    def __init__(self):
        super(MinimalCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=0, bias=True)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(13 * 13 * 4, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))

        x = x.view(x.size(0), -1)

        x = self.fc1(x)

        return x

In [ ]:
def train_model():

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

    model = MinimalCNN()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(10):
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, targets) in enumerate(train_loader):
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

            if batch_idx % 200 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

        accuracy = 100 * correct / total
        print(f'Epoch {epoch+1} - Training Accuracy: {accuracy:.2f}%')

    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, targets in test_loader:
            outputs = model(data)
            test_loss += criterion(outputs, targets).item()
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    test_accuracy = 100 * correct / total
    print(f'\nTest Accuracy: {test_accuracy:.2f}%')

    return model, test_accuracy


In [ ]:

def quantize_weights(model, bits=8):

    quantized_model = MinimalCNN()

    with torch.no_grad():
        conv_weight = model.conv1.weight.data
        conv_bias = model.conv1.bias.data

        weight_scale = (2**(bits-1) - 1) / torch.max(torch.abs(conv_weight))
        bias_scale = (2**(bits-1) - 1) / torch.max(torch.abs(conv_bias))

        quantized_model.conv1.weight.data = torch.round(conv_weight * weight_scale) / weight_scale
        quantized_model.conv1.bias.data = torch.round(conv_bias * bias_scale) / bias_scale

        fc_weight = model.fc1.weight.data
        fc_bias = model.fc1.bias.data

        fc_weight_scale = (2**(bits-1) - 1) / torch.max(torch.abs(fc_weight))
        fc_bias_scale = (2**(bits-1) - 1) / torch.max(torch.abs(fc_bias))

        quantized_model.fc1.weight.data = torch.round(fc_weight * fc_weight_scale) / fc_weight_scale
        quantized_model.fc1.bias.data = torch.round(fc_bias * fc_bias_scale) / fc_bias_scale

    return quantized_model, {
        'conv_weight_scale': weight_scale.item(),
        'conv_bias_scale': bias_scale.item(),
        'fc_weight_scale': fc_weight_scale.item(),
        'fc_bias_scale': fc_bias_scale.item()
    }


In [ ]:


def export_weights_for_systemverilog(model, scales):

    conv_weights = model.conv1.weight.data.numpy()
    conv_bias = model.conv1.bias.data.numpy()

    print(f"// Convolutional Layer Weights (4 filters, 3x3 each)")
    print(f"// Shape: {conv_weights.shape}")
    for i in range(conv_weights.shape[0]):  # For each output channel
        print(f"// Filter {i}")
        filter_weights = conv_weights[i, 0, :, :].flatten()
        weights_str = ", ".join([f"{w:.6f}" for w in filter_weights])
        print(f"parameter real CONV_WEIGHT_{i}[8:0] = '{{{weights_str}}};")

    print(f"\n// Convolutional Layer Biases")
    bias_str = ", ".join([f"{b:.6f}" for b in conv_bias])
    print(f"parameter real CONV_BIAS[3:0] = '{{{bias_str}}};")

    # Export FC layer weights (too large to print all, show structure)
    fc_weights = model.fc1.weight.data.numpy()
    fc_bias = model.fc1.bias.data.numpy()

    print(f"\n// Fully Connected Layer")
    print(f"// Weight matrix shape: {fc_weights.shape}")
    print(f"// This would require {fc_weights.size} weight parameters")
    print(f"// Consider using a smaller intermediate layer or further quantization")

    print(f"\n// FC Layer Biases")
    fc_bias_str = ", ".join([f"{b:.6f}" for b in fc_bias])
    print(f"parameter real FC_BIAS[9:0] = '{{{fc_bias_str}}};")

    print(f"\n// Quantization scales for hardware implementation:")
    for key, value in scales.items():
        print(f"// {key}: {value}")



In [ ]:
def main():
    model, accuracy = train_model()
    quantized_model, scales = quantize_weights(model)
    export_weights_for_systemverilog(quantized_model, scales)
if __name__ == "__main__":
    main()

Epoch 1, Batch 0, Loss: 2.3189
Epoch 1, Batch 200, Loss: 0.3262
Epoch 1, Batch 400, Loss: 0.2476
Epoch 1, Batch 600, Loss: 0.2969
Epoch 1, Batch 800, Loss: 0.1669
Epoch 1 - Training Accuracy: 90.05%
Epoch 2, Batch 0, Loss: 0.1575
Epoch 2, Batch 200, Loss: 0.1528
Epoch 2, Batch 400, Loss: 0.1002
Epoch 2, Batch 600, Loss: 0.1170
Epoch 2, Batch 800, Loss: 0.1406
Epoch 2 - Training Accuracy: 95.55%
Epoch 3, Batch 0, Loss: 0.0940
Epoch 3, Batch 200, Loss: 0.1298
Epoch 3, Batch 400, Loss: 0.2234
Epoch 3, Batch 600, Loss: 0.0737
Epoch 3, Batch 800, Loss: 0.1452
Epoch 3 - Training Accuracy: 96.60%
Epoch 4, Batch 0, Loss: 0.1682
Epoch 4, Batch 200, Loss: 0.3194
Epoch 4, Batch 400, Loss: 0.1485
Epoch 4, Batch 600, Loss: 0.1619
Epoch 4, Batch 800, Loss: 0.0822
Epoch 4 - Training Accuracy: 97.20%
Epoch 5, Batch 0, Loss: 0.1984
Epoch 5, Batch 200, Loss: 0.0524
Epoch 5, Batch 400, Loss: 0.0227
Epoch 5, Batch 600, Loss: 0.0292
Epoch 5, Batch 800, Loss: 0.0636
Epoch 5 - Training Accuracy: 97.52%
Epoch